# 03. Dicts

**Tier:** Intermediate. **Why you'd use this:** You count how often each word appears in a short note. Looking up a word should stay fast as the note grows, and a missing word should not crash your loop.

**After this lesson you can:** count items by key; explain why lookup stays fast as the dict grows; choose `get` when a missing key is normal.
About 30 minutes. **Uses:** [Loops](../03-control-flow/02-loops-lesson.ipynb), [Tuples](02-tuples-lesson.ipynb)


## Quick review

1. Can a list be a dict key? Can a tuple of strings?
2. How do you "change" one field of a tuple?

<details><summary>Answers</summary>

A list cannot; a tuple of strings can. Build a new tuple with the updated field.
</details>


## The idea

A dict maps a key to a value. `data[key]` returns the value and raises `KeyError` when the key is absent. `get` returns a default instead. Keys are inserted in an order the dict remembers.

<details><summary>Explain more</summary>

`data[key] = value` stores or replaces the value for that key. Keys must be hashable: numbers, strings, tuples of those. A missing key is not the same as a key whose value is `0`.
</details>


## See it work

Count how many times each word appears.


In [1]:
def counts(words):
    # 1. Start with no words counted.
    tally = {}
    # 2. One step per word. get supplies 0 the first time.
    for word in words:
        tally[word] = tally.get(word, 0) + 1
    return tally


print(counts(["a", "b", "a"]))
print(counts([]))


{'a': 2, 'b': 1}
{}


## Check yourself

Answer these from the example above before you run the cell.


1. Why can `(1, 2)` be a key but `[1, 2]` cannot?
2. What would go wrong if a key's hash changed after it was inserted?

<details><summary>Answers</summary>

The tuple cannot change, so the slot computed from its hash stays correct. A changed hash sends the next lookup to a different slot, and the entry is still sitting in the old one.
</details>


In [2]:
counts = {}
words = ["to", "be", "or", "not", "to", "be"]
for word in words:
    counts[word] = counts.get(word, 0) + 1
print(counts)
print(list(counts))
print("to" in counts, "to" in words)


{'to': 2, 'be': 2, 'or': 1, 'not': 1}
['to', 'be', 'or', 'not']
True True


<details><summary>Check your answers</summary>

The counts are `to` twice, `be` twice, then `or` and `not` once. `list(counts)` is the keys in the order they were first inserted, not sorted. Both `in` tests are true. They are not the same kind of search.
</details>


## Why it works this way

A dict is a hash table. To store a key, Python calls `hash(key)`, turns that number into a slot in an internal array, and puts the key and the value there. To look a key up, it hashes again and jumps to that slot. It does not walk the entries. Lookup takes about the same time with 10 keys or 10 million, which is average O(1). A bad hash or a huge pile of collisions can make one lookup slow. The average assumes keys scatter. A list has no such jump: `"to" in words` compares against each item, which is O(n).

That speed has a requirement. A key's hash must not change while it is in the dict, or the next lookup jumps to the wrong slot. Keys must be hashable. For the built-in types that means immutable: numbers, strings, and tuples of those work, and lists do not. The error is `unhashable type: 'list'`.

Dicts remember insertion order. The first time a key is stored fixes its place. Replacing the value does not move it. The layout is a compact array of entries plus a smaller index, which is also why a dict uses more memory than a list of pairs but answers lookups without scanning. Deep dive: [How dict and set work](../../internals/04-dicts/01-hash-table-lesson.ipynb).

| operation | list | dict | set |
| --- | --- | --- | --- |
| add at the end, or add a key | amortized O(1) `append` | average O(1) | average O(1) |
| `in` | O(n) scan | average O(1) | average O(1) |
| `insert(0)` | O(n) shift | no position 0 | no position 0 |
| read by index | O(1) | not indexed | not indexed |


In [3]:
import timeit

print(
    "list",
    timeit.timeit(
        "99999 in items",
        setup="items = list(range(100000))",
        number=200,
    ),
)
print(
    "dict",
    timeit.timeit(
        "99999 in lookup",
        setup="lookup = dict.fromkeys(range(100000))",
        number=200,
    ),
)


list 0.160024707991397
dict 5.749985575675964e-06


## When to use it

Use a dict to look things up by a key, to count, or to group. Use a list when you care about position and you scan everything anyway. Use a set when you only need membership, not a value beside the key. Use `get(key, default)` when a missing key is normal. Use `data[key]` when a missing key is a bug you want to hear about.


## Common mistake

`dict.fromkeys` stores the same default object under every key. Appending to one list appends to all of them, because there is only one list.


In [4]:
groups = dict.fromkeys(["a", "b"], [])
groups["a"].append(1)
print(groups)


{'a': [1], 'b': [1]}


Build a new list for each key. `fromkeys` copied the pointer, not the list. The alias is the same sharing as two names for one list.


In [5]:
groups = {}
for key in ["a", "b"]:
    groups[key] = []
groups["a"].append(1)
print(groups)


{'a': [1], 'b': []}


A bare lookup of a missing key raises. Read the last line first: `KeyError`. The arrow marks the lookup. Use `get` when absence is normal.


In [6]:
def value(data, key):
    return data[key]


print(value({"a": 1}, "z"))


KeyError: 'z'

## Try it

Change `counts` so it counts words without caring about capitals. Which line changes, and why don't you lowercase the keys already stored in the dict?


## Recap

- A dict hashes a key and jumps to a slot. Average lookup is O(1). A list scans.
- Keys must stay hashable. Insertion order is the order of first insert.
- `get` supplies a default. `fromkeys` with one list shares that list.

Next: [practice](03-dicts-practice.ipynb).
